In [1]:
import sys
sys.path.append('/opt/micromamba/lib/python3.12/site-packages')

In [2]:
# Vertex SDK for Python
! pip3 install --upgrade --quiet  google-cloud-aiplatform

In [3]:
PROJECT_ID = "project-65113c9e-530b-4341-830"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}
BUCKET_URI = f"gs://mlops-course-project-65113c9e-530b-4341-830-unique-week1"

In [4]:
import os
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

/opt/micromamba/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [19]:
!gsutil cp iris.csv {BUCKET_URI}/iris.csv
!gsutil cp data_v1.csv {BUCKET_URI}/iris_v1.csv
!gsutil cp data_v2.csv {BUCKET_URI}/iris_v2.csv

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://iris.csv [Content-Type=text/csv]...
/ [1 files][  3.8 KiB/  3.8 KiB]                                                
Operation completed over 1 objects/3.8 KiB.                                      
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://data_v1.csv [Content-Type=text/csv]...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      
Google recommends using Gcloud storage CLI (https:/

In [6]:
!gsutil cp {BUCKET_URI}/iris.csv ./iris.csv


Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://mlops-course-project-65113c9e-530b-4341-830-unique-week1/iris.csv...
/ [1 files][  3.8 KiB/  3.8 KiB]                                                
Operation completed over 1 objects/3.8 KiB.                                      


In [7]:
import pandas as pd
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import joblib
from datetime import datetime
import os

In [8]:
data = pd.read_csv('./local_iris.csv')

In [9]:
train, test = train_test_split(data, test_size=0.4, stratify=data['species'], random_state=42)

In [10]:
X_train = train[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y_train = train['species']

In [11]:
mod_dt = DecisionTreeClassifier(max_depth=10, random_state=1)
mod_dt.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=10, random_state=1)

In [12]:
timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
artifact_gcs_path = f"{BUCKET_URI}/artifacts/{timestamp}"

joblib.dump(mod_dt, "artifacts/model.joblib")
!gsutil cp artifacts/model.joblib {artifact_gcs_path}/model.joblib

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://artifacts/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  3.1 KiB/  3.1 KiB]                                                
Operation completed over 1 objects/3.1 KiB.                                      


In [13]:
timestamp

'20260623T145051'

In [14]:
!gsutil cp {artifact_gcs_path}/model.joblib ./downloaded_model.joblib

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://mlops-course-project-65113c9e-530b-4341-830-unique-week1/artifacts/20260623T145051/model.joblib...
/ [1 files][  3.1 KiB/  3.1 KiB]                                                
Operation completed over 1 objects/3.1 KiB.                                      


In [15]:
loaded_model = joblib.load("./downloaded_model.joblib")

In [16]:
def evaluate_dataset(file_path, model):
    df = pd.read_csv(file_path)
    
    X_eval = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
    y_eval = df['species']
    
    predictions = model.predict(X_eval)
    
    acc = accuracy_score(y_eval, predictions)
    print(f"Accuracy: {acc:.3f}\n")

In [17]:
evaluate_dataset("data_v1.csv", loaded_model)
evaluate_dataset("data_v2.csv", loaded_model)

Accuracy: 0.980

Accuracy: 1.000

